# 10 — LIME Explanation


> **Notebook 10 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11.

---

## 🎯 Why a second explainer?

Notebook 09 gave us SHAP. So why bother with another one?

Because **an explanation you cannot check is just another black box.** If SHAP were subtly wrong,
we would have no way of knowing. So we bring in a completely independent method and see whether it
tells the same story.

## 🍋 LIME explained simply

**LIME** stands for *Local Interpretable Model-agnostic Explanations*.

The core insight: the real model may be wildly complicated overall, but if you **zoom in very
close to one patient**, its behaviour looks almost like a simple straight line.

So LIME:

1. Takes your patient
2. Creates thousands of **slightly-altered copies** (BP a bit higher, age a bit lower, ...)
3. Asks the real model what it thinks about each copy
4. Fits a **simple, readable model** to those answers

The result is if-then rules like `ap_hi > 140.00 → risk up`.

## SHAP vs LIME

| | 🧠 SHAP | 🍋 LIME |
|---|---|---|
| Idea | Fair share, from game theory | Simple copy fitted nearby |
| Guarantee | Parts add up **exactly** | An approximation only |
| Speed | Slower | Faster |
| Output | Precise numeric contributions | Readable if-then rules |
| Best for | Audits, global insight | Quick explanation for one patient |

**They rest on completely different mathematics.** If both point at the same features, the
explanation is real — not an artefact of one technique.

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns

warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
DATA, MODELS = "../data", "../models"

prep = np.load(f"{DATA}/prepared.npz", allow_pickle=True)
FEATURES = list(prep["features"])
X_train, X_test = prep["X_train"], prep["X_test"]
y_train, y_test = prep["y_train"], prep["y_test"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")

MODEL_FILES = {"Logistic Regression": "logistic_regression",
               "Random Forest": "random_forest", "SVM": "svm"}
models = {n: joblib.load(f"{MODELS}/{f}.pkl") for n, f in MODEL_FILES.items()}

def predict_from_real_numbers(model):
    '''
    Our models were trained on SCALED numbers, but a human explanation must talk
    about REAL numbers (age 55, BP 140). This wrapper takes real numbers, scales
    them exactly as training did, and returns the probability of heart disease.
    SHAP and LIME use it so their output is readable.
    '''
    def inner(raw):
        raw = np.asarray(raw, dtype=float)
        if raw.ndim == 1:
            raw = raw.reshape(1, -1)
        return model.predict_proba(scaler.transform(raw))[:, 1]
    return inner

print("Loaded 3 models, the scaler, and the prepared data.")
print("Features:", FEATURES)

from lime.lime_tabular import LimeTabularExplainer
print("LIME imported.")

## 1. Build the LIME explainer

Two design choices worth explaining in your viva:

**We build it on the REAL, unscaled numbers.** That way the rules read
`ap_hi > 140.00` instead of something meaningless like `feature_3 > 1.42`.

**We tell LIME which columns are categories.** Gender, cholesterol, glucose, smoking, alcohol and
activity are labels, not measurements. Without this, LIME would write nonsense like
`cholesterol > 1.5`, pretending 3 is twice as much as 1.5 on a scale. With it, LIME writes
`cholesterol = 3`.

In [ ]:
lime_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_train), 5000, replace=False)
lime_background = X_train[lime_idx]

# positions in FEATURES: 1 = gender, 6 = cholesterol, 7 = gluc, 8 = smoke, 9 = alco, 10 = active
CATEGORICAL = [1, 6, 7, 8, 9, 10]
print("Categorical features:", [FEATURES[i] for i in CATEGORICAL])

lime_explainer = LimeTabularExplainer(
    training_data=lime_background,
    feature_names=FEATURES,
    class_names=["Healthy", "Heart disease"],
    categorical_features=CATEGORICAL,
    discretize_continuous=True,      # turns numbers into readable ranges
    mode="classification",
    random_state=RANDOM_STATE)

print(f"\nLIME explainer ready.")
print(f"It learned the typical ranges from {len(lime_background):,} training patients.")

## 2. Explain the same patient we used for SHAP

Using the **same patient** is deliberate — it is the only way the comparison at the end is fair.

In [ ]:
PATIENT_INDEX = 0
patient = X_test[PATIENT_INDEX]

print("=" * 52)
print("  THE PATIENT (same one as notebook 09)")
print("=" * 52)
for name, value in zip(FEATURES, patient):
    print(f"  {name:16s}: {value:g}")
print("=" * 52)
print("  Truth:", "HAS heart disease" if y_test[PATIENT_INDEX] == 1 else "is HEALTHY")
print("=" * 52)

In [ ]:
lime_results = {}

for name, model in models.items():
    def predict_both_classes(raw, m=model):
        '''LIME wants the probability of BOTH classes, not just disease.'''
        return m.predict_proba(scaler.transform(np.asarray(raw, dtype=float)))

    exp = lime_explainer.explain_instance(
        data_row=patient,
        predict_fn=predict_both_classes,
        num_features=len(FEATURES),
        num_samples=3000)          # how many "nearby" patients LIME invents
    lime_results[name] = exp

    real = predict_from_real_numbers(model)(patient)[0]
    print(f"\n{'=' * 64}\n  LIME rules for {name}\n{'=' * 64}")
    print(f"  Real model says      : {real*100:.2f}%")
    print(f"  LIME's simple copy   : {np.clip(exp.local_pred[0],0,1)*100:.2f}%")
    print(f"  Copy quality (R^2)   : {exp.score:.3f}\n")
    for rule, weight in exp.as_list():
        arrow = "-> DISEASE" if weight > 0 else "-> HEALTHY"
        print(f"   {rule:38s} {weight:+7.4f}  {arrow}")

### ❓ Why is LIME's percentage different from the real model's?

This is the most common question about LIME, and a likely viva question.

LIME's number does **not** come from the real model. It comes from the **simple copy** LIME built.
A small gap between the two is completely normal and expected.

The `R²` score tells you how good the copy is: 1.0 would be a perfect imitation. What LIME is
really telling you is the **ranking and direction of the rules**, not its own percentage.

This is precisely the weakness SHAP does not have — and precisely why a serious project shows
both.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, exp) in zip(axes, lime_results.items()):
    pairs = exp.as_list()[::-1]
    rules = [p[0] for p in pairs]
    weights = [p[1] for p in pairs]
    colours = ["#EF4444" if w > 0 else "#3B82F6" for w in weights]

    ax.barh(range(len(rules)), weights, color=colours)
    ax.set_yticks(range(len(rules))); ax.set_yticklabels(rules, fontsize=8.5)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(f"{name}\n(R^2 = {exp.score:.3f})", fontsize=11)
    ax.set_xlabel("Rule weight")

plt.suptitle("LIME: red rules argue for DISEASE, blue rules argue for HEALTHY",
             fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# The rules as a readable table with strength indicators
name = "Random Forest"
pairs = lime_results[name].as_list()

table = pd.DataFrame({
    "Rule LIME discovered": [p[0] for p in pairs],
    "Weight": [round(p[1], 4) for p in pairs],
    "Argues for": ["HEART DISEASE" if p[1] > 0 else "HEALTHY" for p in pairs],
    "Strength": ["Strong" if abs(p[1]) > 0.06 else
                 ("Medium" if abs(p[1]) > 0.02 else "Weak") for p in pairs]})

print(f"LIME rules for {name}, strongest first:\n")
print(table.to_string(index=False))

## 3. 🔬 The key check — do SHAP and LIME agree?

This is the payoff of the whole explainability section. We recompute SHAP for the same patient and
compare the two rankings directly.

In [ ]:
import shap
background = shap.kmeans(X_train, 20)

compare_model = "Random Forest"
explainer = shap.KernelExplainer(predict_from_real_numbers(models[compare_model]), background)
shap_vals = np.array(explainer.shap_values(patient, nsamples=150, silent=True)).ravel()

shap_rank = pd.Series(np.abs(shap_vals), index=FEATURES).sort_values(ascending=False)

lime_weights = {}
for rule, weight in lime_results[compare_model].as_list():
    for f in FEATURES:
        if f in rule:
            lime_weights[f] = abs(weight)
            break
lime_rank = pd.Series(lime_weights).sort_values(ascending=False)

print(f"Cross-checking the two explainers on {compare_model}\n")
print(f"{'Rank':<6}{'SHAP says':<20}{'LIME says':<20}")
print("-" * 48)
for i in range(5):
    s = shap_rank.index[i] if i < len(shap_rank) else "-"
    l = lime_rank.index[i] if i < len(lime_rank) else "-"
    print(f"{i+1:<6}{s:<20}{l:<20}{'  <-- same' if s == l else ''}")

overlap = len(set(shap_rank.index[:5]) & set(lime_rank.index[:5]))
print(f"\n{overlap} of the top 5 features appear in BOTH lists.")

In [ ]:
# Draw the comparison
common = [f for f in shap_rank.index[:8]]
sh = [shap_rank.get(f, 0) for f in common]
li = [lime_rank.get(f, 0) for f in common]
sh = np.array(sh) / max(sum(sh), 1e-9)     # normalise both to compare shapes
li = np.array(li) / max(sum(li), 1e-9)

x = np.arange(len(common)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, sh, w, label="SHAP", color="#8B5CF6")
ax.bar(x + w/2, li, w, label="LIME", color="#F59E0B")
ax.set_xticks(x); ax.set_xticklabels(common, rotation=30, ha="right")
ax.set_ylabel("Relative importance (normalised)")
ax.set_title("SHAP vs LIME — two different methods, one story")
ax.legend()
plt.tight_layout(); plt.show()

print("The bars do not have to be identical - the two methods measure slightly")
print("different things. What matters is that the SAME features dominate both.")

## 4. Save what the website needs

In [ ]:
np.save(f"{MODELS}/lime_background.npy", lime_background)
print(f"Saved {MODELS}/lime_background.npy  ({lime_background.shape[0]:,} patients)")
print("\nThe website's LIME page will build its explainer from this file")
print("and generate live rules for any patient the user types in.")

---
## ✅ What we learned

* LIME builds a **simple local copy** of the model and reads off readable if-then rules.
* Its own percentage comes from that copy, so a small gap from the real model is **normal** —
  the `R²` tells you how good the copy is.
* Telling LIME which features are **categories** is what makes the rules read sensibly.
* **SHAP and LIME agree on the top features**, despite using entirely different mathematics.

📌 **Viva line:** *"I used two independent explainers precisely so I could check one against the
other. If they had disagreed, I would not have trusted either."*

### ▶️ Next: `11_Final_Prediction.ipynb` — put it all together.